In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SILVER_CLEAN = Path("../data/silver/transactions_clean.parquet")
df = pd.read_parquet(SILVER_CLEAN)

print("Shape Silver clean:", df.shape)
df.head(3)

Shape Silver clean: (1048575, 13)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_diff_orig,balance_diff_dest
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,0.0,9839.64
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,0.0,1864.28
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,0.0,181.00


In [2]:
# Optimisation mémoire (utile avec 1M+ lignes)
df["type"] = df["type"].astype

In [4]:
tx = df.copy()

# 1) Flags sur le type de transaction
tx["is_transfer"] = (tx["type"] == "TRANSFER").astype(int)
tx["is_cash_out"] = (tx["type"] == "CASH_OUT").astype(int)
tx["is_cash_in"]  = (tx["type"] == "CASH_IN").astype(int)
tx["is_payment"]  = (tx["type"] == "PAYMENT").astype(int)
tx["is_debit"]    = (tx["type"] == "DEBIT").astype(int)

# 2) Log du montant
tx["amount_log"] = np.log1p(tx["amount"])

# 3) Montants élevés (percentiles)
p95 = tx["amount"].quantile(0.95)
p99 = tx["amount"].quantile(0.99)
tx["is_large_p95"] = (tx["amount"] >= p95).astype(int)
tx["is_large_p99"] = (tx["amount"] >= p99).astype(int)

# 4) Balance diffs (si elles n’existent pas, on les calcule)
if "balance_diff_orig" not in tx.columns:
    tx["balance_diff_orig"] = tx["oldbalanceOrg"] - tx["amount"] - tx["newbalanceOrig"]

if "balance_diff_dest" not in tx.columns:
    tx["balance_diff_dest"] = tx["oldbalanceDest"] + tx["amount"] - tx["newbalanceDest"]

# 5) Incohérences (flags)
tx["abs_balance_diff_orig"] = tx["balance_diff_orig"].abs()
tx["abs_balance_diff_dest"] = tx["balance_diff_dest"].abs()

tx["is_incoherent_orig"] = (tx["abs_balance_diff_orig"] > 1e-6).astype(int)
tx["is_incoherent_dest"] = (tx["abs_balance_diff_dest"] > 1e-6).astype(int)

# 6) Features relatives (montant vs balance avant) - safe (évite division par 0)
tx["orig_balance_before"] = tx["oldbalanceOrg"].replace(0, np.nan)
tx["dest_balance_before"] = tx["oldbalanceDest"].replace(0, np.nan)

tx["orig_rel_amount"] = (tx["amount"] / tx["orig_balance_before"]).fillna(0)
tx["dest_rel_amount"] = (tx["amount"] / tx["dest_balance_before"]).fillna(0)

# 7) (IMPORTANT) Garder la target à part (évite fuite)
# isFraud = target, pas une feature
target = tx["isFraud"].copy()

# Dataset features sans la target
tx_features = tx.drop(columns=["isFraud"])

print("✅ tx_features:", tx_features.shape, "| target:", target.shape)
print("Fraud rate (target.mean):", target.mean())


✅ tx_features: (1048575, 28) | target: (1048575,)
Fraud rate (target.mean): 0.0010890971079798775


In [5]:
cust = (
    tx_features
    .groupby("nameOrig")
    .agg(
        tx_count=("amount", "size"),
        total_amount=("amount", "sum"),
        mean_amount=("amount", "mean"),
        std_amount=("amount", "std"),
        max_amount=("amount", "max"),

        n_transfer=("is_transfer", "sum"),
        n_cash_out=("is_cash_out", "sum"),
        n_large_p99=("is_large_p99", "sum"),

        unique_dest=("nameDest", "nunique"),

        incoh_orig_count=("is_incoherent_orig", "sum"),
        incoh_dest_count=("is_incoherent_dest", "sum"),
    )
    .reset_index()
)

# std_amount peut être NaN si un client n'a qu'une transaction
cust["std_amount"] = cust["std_amount"].fillna(0)

# Ratios (comportement)
cust["ratio_transfer"] = cust["n_transfer"] / cust["tx_count"]
cust["ratio_cash_out"] = cust["n_cash_out"] / cust["tx_count"]
cust["ratio_large_p99"] = cust["n_large_p99"] / cust["tx_count"]

cust["ratio_incoh_orig"] = cust["incoh_orig_count"] / cust["tx_count"]
cust["ratio_incoh_dest"] = cust["incoh_dest_count"] / cust["tx_count"]

print("✅ customers features:", cust.shape)
cust.head(5)


✅ customers features: (1048317, 17)


,nameOrig,tx_count,total_amount,mean_amount,std_amount,max_amount,n_transfer,n_cash_out,n_large_p99,unique_dest,incoh_orig_count,incoh_dest_count,ratio_transfer,ratio_cash_out,ratio_large_p99,ratio_incoh_orig,ratio_incoh_dest
0,C1000001725,1,8424.74,8424.74,0.0,8424.74,0,0,0,1,1,1,0.0,0.0,0.0,1.0,1.0
1,C1000004530,1,93865.13,93865.13,0.0,93865.13,0,0,0,1,0,1,0.0,0.0,0.0,0.0,1.0
2,C1000008582,1,315626.96,315626.96,0.0,315626.96,0,0,0,1,1,0,0.0,0.0,0.0,1.0,0.0
3,C1000009135,1,3849.38,3849.38,0.0,3849.38,0,0,0,1,0,1,0.0,0.0,0.0,0.0,1.0
4,C1000012640,1,367527.28,367527.28,0.0,367527.28,0,0,0,1,1,1,0.0,0.0,0.0,1.0,1.0


In [6]:
GOLD_DIR = Path("../data/gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

TX_GOLD_PATH = GOLD_DIR / "transactions_gold.parquet"
CUST_GOLD_PATH = GOLD_DIR / "customers_gold.parquet"

tx_features.to_parquet(TX_GOLD_PATH, index=False)
cust.to_parquet(CUST_GOLD_PATH, index=False)

print(" Saved:", TX_GOLD_PATH)
print(" Saved:", CUST_GOLD_PATH)


ArrowInvalid: ('Could not convert <bound method NDFrame.astype of 0           PAYMENT\n1           PAYMENT\n2          TRANSFER\n3          CASH_OUT\n4           PAYMENT\n             ...   \n1048570    CASH_OUT\n1048571     PAYMENT\n1048572     PAYMENT\n1048573     PAYMENT\n1048574     PAYMENT\nName: type, Length: 1048575, dtype: object> with type method: did not recognize Python value type when inferring an Arrow data type', 'Conversion failed for column type with type object')

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

# 1) Créer le dossier gold
GOLD_DIR = Path("../data/gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

TX_GOLD_PATH = GOLD_DIR / "transactions_gold.parquet"
CUST_GOLD_PATH = GOLD_DIR / "customers_gold.parquet"

# 2) Fonction de "nettoyage des types" pour éviter ArrowInvalid
def make_parquet_safe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Convertir toutes les colonnes category -> string (souvent plus stable)
    cat_cols = out.select_dtypes(include=["category"]).columns
    for c in cat_cols:
        out[c] = out[c].astype("string")

    # Convertir colonnes object -> string (évite types mélangés)
    obj_cols = out.select_dtypes(include=["object"]).columns
    for c in obj_cols:
        out[c] = out[c].astype("string")

    # Forcer les colonnes numériques (au cas où une colonne est en string)
    # On essaye de convertir tout ce qui ressemble à du numérique
    for c in out.columns:
        if c not in obj_cols and c not in cat_cols:
            # Si c'est déjà numérique, on laisse
            continue

    # Remplacer inf/-inf (parquet n'aime pas parfois)
    out = out.replace([np.inf, -np.inf], np.nan)

    return out

# 3) Appliquer sur les 2 datasets
tx_features_safe = make_parquet_safe(tx_features)
cust_safe = make_parquet_safe(cust)

# 4) Sauvegarder en Parquet (avec pyarrow)
tx_features_safe.to_parquet(TX_GOLD_PATH, index=False, engine="pyarrow")
cust_safe.to_parquet(CUST_GOLD_PATH, index=False, engine="pyarrow")

print("✅ Saved:", TX_GOLD_PATH)
print("✅ Saved:", CUST_GOLD_PATH)
print("Transactions Gold shape:", tx_features_safe.shape)
print("Customers Gold shape:", cust_safe.shape)

✅ Saved: ..\data\gold\transactions_gold.parquet
✅ Saved: ..\data\gold\customers_gold.parquet
Transactions Gold shape: (1048575, 28)
Customers Gold shape: (1048317, 17)
